In [115]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [ ]:
AWS_ACCESS_KEY_ID = "xxxxxxxxxxxxxxx"
AWS_SECRET_ACCESS_KEY = "xxxxxxxxxxxxxxx"
# AWS_REGION = "eu-central-1"

INPUT_PATH = "s3a://your-bucket-name/your-data-folder/"



spark = (
    SparkSession.builder.appName("SparkPartitionConcept")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.executor.instances", "1")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    # .config("spark.sql.files.maxPartitionBytes", 213909504)
    .getOrCreate()
)

orders = spark.read.parquet(INPUT_PATH)

In [6]:
orders.show(5)

+-----------+----------+----------+---------------+--------+----------+--------------+----------+--------------------+--------------------+-------+----+-----+---+
|shipment_id|carrier_id|origin_hub|destination_hub|route_id|weight_lbs|distance_miles|    status|          created_ts|          updated_ts|sla_met|year|month|day|
+-----------+----------+----------+---------------+--------+----------+--------------+----------+--------------------+--------------------+-------+----+-----+---+
|  108000000|  CAR-0058|       KCK|            DET| KCK-DET|   9209.63|         662.7|   DELAYED|2026-09-21 17:36:...|2026-09-22 02:03:...|   true|2026|    9| 21|
|  108000001|  CAR-0140|       DET|            CLE| DET-CLE|  28864.92|         332.4| CANCELLED|2026-09-21 07:34:...|2026-09-21 15:35:...|   true|2026|    9| 21|
|  108000002|  CAR-0088|       MKE|            CIN| MKE-CIN|    2948.6|         885.7| CANCELLED|2026-09-21 05:41:...|2026-09-21 14:25:...|   true|2026|    9| 21|
|  108000003|  CAR-012

In [117]:
# Get the number of partitions for the DataFrame

orders.rdd.getNumPartitions()

30

It shows 30 as the partition count for the entire 3.4 GB parquet file from s3.

- If the below script runs , this will show 30 tasks , since one 1 task can use 1 CPU to process 1 partition at a time.

In [118]:
orders.filter(orders["status"] == "DELAYED").count()

26999221

In [70]:
orders.count()

162000000

### REPARTITION

Repartition: This is a spark method used to recreate a partition on `spark DataFrame`.
- `Repartition()` triggers `shuffle` and affect performance of your spark application, avoid it as much as possible.
- Best practice is to use the `spark.sql.files.maxPartitionBytes` configuration to create your preferred number of partition at the initial read , ensure to read the configuration `spark.files.openCostInBytes`, so as to avoid surprise.
- Documenation on Partition Tunning is [HERE](https://spark.apache.org/docs/latest/sql-performance-tuning.html#tuning-partitions)
- Documenation on repartition usage is [HERE](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.repartition.html)

In [103]:
# To see the number of partition for a Dataframe
orders.rdd.getNumPartitions()

30

In [104]:
df_repartition = orders.repartition(18)

In [105]:
# df_repartition.rdd.getNumPartitions()

In [106]:
df_repartition.filter(df_repartition["status"] == "DELAYED").show(5)

[Stage 1:=======================================================> (29 + 1) / 30]

+-----------+----------+----------+---------------+--------+----------+--------------+-------+--------------------+--------------------+-------+----+-----+---+
|shipment_id|carrier_id|origin_hub|destination_hub|route_id|weight_lbs|distance_miles| status|          created_ts|          updated_ts|sla_met|year|month|day|
+-----------+----------+----------+---------------+--------+----------+--------------+-------+--------------------+--------------------+-------+----+-----+---+
|  108969046|  CAR-0072|       CIN|            DET| CIN-DET|  43046.66|         731.0|DELAYED|2026-09-21 22:28:...|2026-09-22 04:10:...|  false|2026|    9| 21|
|  112378916|  CAR-0104|       KCK|            OMA| KCK-OMA|   8479.15|         901.6|DELAYED|2026-09-21 08:12:...|2026-09-21 16:49:...|   true|2026|    9| 21|
|  109278602|  CAR-0025|       MSP|            STL| MSP-STL|   40180.3|         850.2|DELAYED|2026-09-21 05:10:...|2026-09-21 11:03:...|  false|2026|    9| 21|
|  111280389|  CAR-0017|       CIN|     

### COALESCE

`Coalesce` is also a spark method used to `repartition`, but it doesn't `trigger shuffle`.
- `Coalesce()` cannot be used to increase a partition, only reduction.
  - To increase a partition, that can be done using the `repartition()` method.
  - There is a workaround to use coalesce to increase partition, but it triggers shuffle.

Documentation usage reference is [HERE](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.coalesce.html)

In [94]:
df_coalesce = orders.coalesce(15)

In [97]:
df_coalesce.rdd.getNumPartitions()

15

In [98]:
df_coalesce.filter(df_coalesce["status"] == "DELAYED").show(5)

+-----------+----------+----------+---------------+--------+----------+--------------+-------+--------------------+--------------------+-------+----+-----+---+
|shipment_id|carrier_id|origin_hub|destination_hub|route_id|weight_lbs|distance_miles| status|          created_ts|          updated_ts|sla_met|year|month|day|
+-----------+----------+----------+---------------+--------+----------+--------------+-------+--------------------+--------------------+-------+----+-----+---+
|  108000000|  CAR-0058|       KCK|            DET| KCK-DET|   9209.63|         662.7|DELAYED|2026-09-21 17:36:...|2026-09-22 02:03:...|   true|2026|    9| 21|
|  108000034|  CAR-0024|       KCK|            MSP| KCK-MSP|  14881.33|         413.6|DELAYED|2026-09-21 05:05:...|2026-09-21 15:45:...|   true|2026|    9| 21|
|  108000043|  CAR-0123|       CLE|            IND| CLE-IND|   2103.59|         445.7|DELAYED|2026-09-21 14:46:...|2026-09-22 01:45:...|   true|2026|    9| 21|
|  108000050|  CAR-0100|       KCK|     

### COALESCE USECASE

In [109]:
delayed_df = orders.filter(orders["status"] == "DELAYED")

single_file_df = delayed_df.coalesce(1)

print("Partitions after coalesce:", single_file_df.rdd.getNumPartitions())  

# 3. Write to S3 
output_path = "s3a://spark-apex-freight-data/curated/delayed_2026_11_23/"

(
    single_file_df.write
    .mode("overwrite")
    .parquet(output_path)
)

Partitions after coalesce: 1


Tradeoff
- This took 5.5 mins to write 1 file to s3
- Only 1 Core doing the  whole work of
  - Reading the dataframe
  - Filtering
  - Parquet Compression
  - Uploading to S3

In [113]:
delayed_df = orders.filter(orders["status"] == "DELAYED")

single_file_df = delayed_df.coalesce(12)

print("Partitions after coalesce:", single_file_df.rdd.getNumPartitions())  

# 3. Write to S3 
output_path = "s3a://spark-apex-freight-data/curated/delayed_2026_11_23/"

(
    single_file_df.write
    .mode("overwrite")
    .parquet(output_path)
)

Partitions after coalesce: 12


In [114]:
spark.stop()

Please read on this 2 important configuration on Partition Tuning.
- maxPartitionBytes
- openCostInBytes

The Documenation reference is [HERE](https://spark.apache.org/docs/latest/sql-performance-tuning.html#tuning-partitions)

Spark Configurations can be found [HERE](https://spark.apache.org/docs/latest/configuration.html)